# Neste notebook vamos fazer o tratamento e limpeza dos dados.

## Analise de Dados de Airbnb no Rio de Janeiro 

Fonte: Inside Airbnb - Rio de Janeiro - https://insideairbnb.com/rio-de-janeiro/

## 1. Carregamento de Dados

In [ ]:
# importando bibliotecas 
import polars as pl
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import numpy as np

# remover avisos desnecessarios
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# carrega os dados
df = pd.read_csv('listings_rio_de_janeiro_cleaned.csv')

## 2. Limpeza e tratamento dos dados.

In [ ]:
# visualizando as 5 primeiras linhas
df.head()

In [ ]:
# visualizando as 5 ultimas linhas
df.tail()

In [ ]:
# visualizando os tipos de dados
df.dtypes

In [ ]:
# visualizando os nulos
df.isnull().sum().sort_values(ascending=False)

In [ ]:
# preenchendo valores nulos
df = df.fillna({'num_bathrooms': 0, 'bathrooms': 0})

In [ ]:
# verificando linhas duplicadas
df.drop(columns='neighbourhood_cleansed').duplicated().sum()

In [ ]:
# removendo linhas duplicadas 
df.drop_duplicates()

In [ ]:
# verificando localiza??o exata
colunas_geo = ['latitude', 'longitude']

total_duplicados_geo = df.duplicated(subset=colunas_geo).sum()

print(f"Existem {total_duplicados_geo} an?ncios que compartilham a mesma localiza??o exata.")


In [ ]:
# filtrando o DataFrame para mostrar apenas as linhas que t?m coordenadas repetidas
anuncios_no_mesmo_lugar = df[df.duplicated(subset=colunas_geo, keep=False)]

# ordena por latitude para ver os pares juntos
display(anuncios_no_mesmo_lugar.sort_values(by='latitude').head(10))


## Decis?o sobre os dados com localiza??o iguais

Identificamos que 4259 an?ncios compartilham as localiza??es entre si. Vamos considerar mais de um im?vel por localiza??o. 

# 3. Renomeando colunas

In [ ]:
# 1. Renomeando colunas principais

df = df.rename(columns={'neighbourhood_cleansed': 'BAIRRO', 
                    'property_type': 'TIPO_IMOVEL',
                    'room_type': 'TIPO_QUARTO',
                    'bathrooms': 'BANHEIROS',
                    'bedrooms': 'QUARTOS',
                    'beds': 'CAMAS',
                    'amenities': 'COMODIDADE',
                    'price': 'PRECO',
                    'minimum_nights': 'NOITES_MINIMAS',
                    'maximum_nights': 'NOITES_MAXIMAS',
                    'accommodates': 'LIMITE_PESSOAS',
                    'review_scores_accuracy': 'MEDIA_AVALIACAO',
                    'review_scores_communication': 'AVALIACAO_COMUNICACAO',
                    'review_scores_location': 'AVALIACAO_LOCALIZACAO',
                    'price_per_person': 'PRECO_POR_PESSOA'
    })

# 2. Renomeando dados dentro das colunas
condicoes = [
    df['TIPO_IMOVEL'].str.contains(r'shared room|shared', case=False, na=False),
    df['TIPO_IMOVEL'].str.contains(r'private room', case=False, na=False),
    df['TIPO_IMOVEL'].str.contains(r'condo|unit|apartment|loft|suit', case=False, na=False),
    df['TIPO_IMOVEL'].str.contains(r'house|villa|entire home', case=False, na=False),
    df['TIPO_IMOVEL'].str.contains(r'room in hotel|Room in hostel', case=False, na=False)
]

escolhas = [
    'Quarto Compartilhado',
    'Quarto Privado',
    'Apartamento',
    'Casa',
    'Hotel/Hostel'
]

df['TIPO_IMOVEL'] = np.select(condicoes, escolhas, default='Outros')

In [ ]:
# Arredondando para 2 casas decimais
df['PRECO_POR_PESSOA'] = df['PRECO_POR_PESSOA'].round(2)

In [ ]:
df.head(10)

# 4. Exportando arquivo tratado


In [ ]:
# No seu primeiro notebook
df.to_parquet('airbnb_rio_tratado.parquet', index=False)